# Notebook 03: Sequence Models and Why Transformers

In notebooks 01 and 02, we built up the math (linear algebra) and mechanics (deep learning) we need. Now we ask a crucial question: **why do we need a new architecture at all?**

This notebook tells the story:

1. **The problem** — why sequences are hard
2. **RNNs** — the first serious idea, and why it works (sort of)
3. **Why RNNs struggle** — sequential bottleneck, vanishing gradients, no parallelism
4. **The attention idea** — "look at everything at once"
5. **A simple attention demo** — dot-product scores, softmax, heatmap
6. **Preview** — what the Transformer paper proposes

This notebook is intentionally more conceptual than the previous two. We want to deeply understand the *motivation* before building the architecture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

## 1. The Problem: Sequences Are Everywhere

Many important types of data come in **sequences** — ordered lists where position matters:

- **Text**: "The cat sat on the mat" (swap words and meaning changes)
- **Time series**: stock prices, sensor readings, weather
- **Audio**: speech is a sequence of sounds
- **DNA**: ATCGGCTA...

The key challenge: **order matters**. Consider these two sentences:

> "The dog bit the man" vs. "The man bit the dog"

Same words, completely different meaning. A model that just bags words together (ignoring order) loses critical information.

What we need: a model that can process a sequence **while keeping track of position and context**.

In [ ]:
# A simple demonstration: order matters
# Let's represent words as random vectors and show that
# summing them (bag-of-words) loses order information

vocab = {"the": 0, "dog": 1, "bit": 2, "man": 3}
embed_dim = 4

# Simple embeddings (random for illustration)
embeddings = np.random.randn(len(vocab), embed_dim)

sentence_a = ["the", "dog", "bit", "the", "man"]  # dog bites man
sentence_b = ["the", "man", "bit", "the", "dog"]  # man bites dog

# Bag-of-words: just sum the embeddings
bow_a = sum(embeddings[vocab[w]] for w in sentence_a)
bow_b = sum(embeddings[vocab[w]] for w in sentence_b)

print("Sentence A (dog bit man):", sentence_a)
print("Sentence B (man bit dog):", sentence_b)
print()
print("Bag-of-words A:", np.round(bow_a, 2))
print("Bag-of-words B:", np.round(bow_b, 2))
print()
print("Are they identical?", np.allclose(bow_a, bow_b))
print("\n=> Bag-of-words CANNOT distinguish these two sentences!")

## 2. The RNN Idea: Process One Token at a Time

The Recurrent Neural Network (RNN) was the dominant approach for sequences before Transformers. The core idea is beautifully simple:

**Maintain a "hidden state" that gets updated as you read each token.**

Think of it like reading a book: you keep a running mental summary, and each new word updates that summary.

```
  Input:     x1        x2        x3        x4
             |         |         |         |
             v         v         v         v
  State: h0 -> [RNN] -> h1 -> [RNN] -> h2 -> [RNN] -> h3 -> [RNN] -> h4
                                                                       |
                                                                       v
                                                                    output
```

At each step:
```
  h_t = f(W_h * h_{t-1} + W_x * x_t + b)
```

- `h_{t-1}` is the previous hidden state (the "memory")
- `x_t` is the current input
- `W_h` and `W_x` are learned weight matrices
- `f` is a nonlinearity (like tanh)

The same weights are **shared** across all time steps — this is what makes it "recurrent."

In [ ]:
# Let's trace through an RNN step by step (no training, just the forward pass)

def rnn_forward(inputs, W_h, W_x, b, h0):
    """Simple RNN forward pass — one step at a time."""
    h = h0
    states = [h]
    for x in inputs:
        h = np.tanh(W_h @ h + W_x @ x + b)
        states.append(h)
    return states

# Setup: 4-dim embeddings, 3-dim hidden state
hidden_dim = 3
W_h = np.random.randn(hidden_dim, hidden_dim) * 0.5
W_x = np.random.randn(hidden_dim, embed_dim) * 0.5
b = np.zeros(hidden_dim)
h0 = np.zeros(hidden_dim)

# Process "the dog bit the man"
inputs_a = [embeddings[vocab[w]] for w in sentence_a]
inputs_b = [embeddings[vocab[w]] for w in sentence_b]

states_a = rnn_forward(inputs_a, W_h, W_x, b, h0)
states_b = rnn_forward(inputs_b, W_h, W_x, b, h0)

print("RNN can distinguish order!")
print(f"Final state for 'dog bit man': {np.round(states_a[-1], 3)}")
print(f"Final state for 'man bit dog': {np.round(states_b[-1], 3)}")
print(f"Same? {np.allclose(states_a[-1], states_b[-1])}")

In [ ]:
# Visualize how the hidden state evolves over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, states, title in [
    (axes[0], states_a, '"the dog bit the man"'),
    (axes[1], states_b, '"the man bit the dog"'),
]:
    states_arr = np.array(states)  # shape: (6, 3) — h0 through h5
    for dim in range(hidden_dim):
        ax.plot(states_arr[:, dim], marker='o', label=f'h[{dim}]')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Hidden state value')
    ax.set_title(f'RNN hidden state: {title}')
    ax.legend()
    ax.set_xticks(range(len(states)))
    ax.set_xticklabels(['h0'] + sentence_a if title.startswith('"the dog') else ['h0'] + sentence_b,
                       rotation=45)

plt.tight_layout()
plt.show()
print("Notice: same words, different order => different hidden state trajectories.")

## 3. Why RNNs Struggle

RNNs work, but they have three serious problems:

### Problem 1: Sequential Bottleneck (Can't Parallelize)

```
  To compute h3, you MUST first compute h1, then h2, then h3.
  
  h0 -> h1 -> h2 -> h3 -> h4 -> h5 -> ... -> h1000
         ↑     ↑     ↑
      can't  can't  can't
      skip   skip   skip
```

With a GPU that has thousands of cores, you'd love to process all positions in parallel. But RNNs force you to go one step at a time. For a 1000-token document, that's 1000 sequential operations.

### Problem 2: Vanishing Gradients (Forgetting the Past)

Information from early tokens must survive through many multiplications to reach later tokens. Imagine passing a message through 100 people — it gets distorted.

### Problem 3: Fixed-Size Bottleneck

The entire history of the sequence gets compressed into a single fixed-size vector `h_t`. For a 500-word paragraph, everything must fit into, say, 512 numbers.

In [ ]:
# Demonstration: vanishing gradients
# Show how a signal fades as it passes through many RNN steps

def signal_decay(seq_length, weight_scale):
    """Simulate how much of the initial signal survives after N steps."""
    # Simplified: just track how gradient magnitude changes
    # In an RNN, gradients multiply by W_h at each step
    W = np.random.randn(3, 3) * weight_scale
    signal = np.ones(3)
    magnitudes = [np.linalg.norm(signal)]
    for _ in range(seq_length):
        signal = np.tanh(W @ signal)  # simplified RNN step
        magnitudes.append(np.linalg.norm(signal))
    return magnitudes

fig, ax = plt.subplots(figsize=(10, 4))
for scale, label in [(0.5, 'Small weights (vanishing)'),
                      (1.0, 'Medium weights'),
                      (2.0, 'Large weights (exploding)')]:
    mags = signal_decay(50, scale)
    ax.plot(mags, label=label, linewidth=2)

ax.set_xlabel('Time steps')
ax.set_ylabel('Signal magnitude')
ax.set_title('The Vanishing/Exploding Gradient Problem')
ax.legend()
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()
print("With small weights, the signal vanishes — the RNN 'forgets' early inputs.")
print("With large weights, the signal explodes — training becomes unstable.")
print("Finding the sweet spot is very difficult in practice.")

In [ ]:
# Demonstration: processing time — RNN vs. parallel
# This is just a conceptual visualization

seq_lengths = [10, 50, 100, 200, 500, 1000]

rnn_time = seq_lengths  # O(n) — must process sequentially
parallel_time = [1] * len(seq_lengths)  # O(1) — process all at once (idealized)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seq_lengths, rnn_time, 'o-', label='RNN (sequential)', linewidth=2)
ax.plot(seq_lengths, parallel_time, 's-', label='Parallel (ideal)', linewidth=2)
ax.set_xlabel('Sequence length')
ax.set_ylabel('Processing steps (relative)')
ax.set_title('Sequential vs. Parallel Processing')
ax.legend()
plt.tight_layout()
plt.show()
print("RNNs scale linearly with sequence length — a huge bottleneck on modern GPUs.")

## 4. The Attention Idea: Look at Everything at Once

What if, instead of reading left-to-right and compressing everything into a single hidden state, we could **look at all positions simultaneously** and decide which ones are relevant?

This is the key insight behind **attention**.

### A Translation Example

Consider translating English to French:

```
  English: "The  black  cat  sat  on  the  mat"
  French:  "Le   chat   noir ..."
```

When generating the French word **"noir"** (black), the model needs to look back at the English word **"black"** — which is at position 2, not the most recent position.

With an RNN, the information about "black" has been compressed into the hidden state and may be faded. With attention, we can **directly attend to position 2**.

### The Attention Mechanism (Intuition)

```
  For each output position:
  1. Compare it with EVERY input position ("How relevant is each input?")
  2. Turn those comparisons into weights (softmax → probabilities)
  3. Take a weighted sum of the inputs
```

The comparison is done with a **dot product** — the same operation we learned in notebook 01. Two vectors that point in similar directions get a high score.

In [ ]:
# Simple attention demo: which English words does each French word attend to?

def softmax(x):
    """Numerically stable softmax."""
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Pretend we have embeddings for English and French words
# We'll craft them so that related words have similar embeddings

english_words = ["The", "black", "cat", "sat", "on", "the", "mat"]
french_words  = ["Le", "chat", "noir", "est", "assis", "sur", "le", "tapis"]

d = 8  # embedding dimension

# Create embeddings where related word pairs have high dot products
np.random.seed(7)
eng_embed = np.random.randn(len(english_words), d) * 0.5
fr_embed  = np.random.randn(len(french_words), d) * 0.5

# Manually boost similarity for related pairs
# (The, Le), (black, noir), (cat, chat), (sat, est/assis), (on, sur), (the, le), (mat, tapis)
pairs = [(0,0), (1,2), (2,1), (3,3), (3,4), (4,5), (5,6), (6,7)]
for eng_i, fr_i in pairs:
    shared = np.random.randn(d)
    eng_embed[eng_i] += shared
    fr_embed[fr_i] += shared

# Compute attention scores: each French word attends to all English words
# scores[i, j] = dot(french[i], english[j])
scores = fr_embed @ eng_embed.T  # shape: (8, 7)

# Apply softmax to get attention weights
attn_weights = softmax(scores)

print("Attention weights (French → English):")
print(f"{'':>8}", '  '.join(f'{w:>6}' for w in english_words))
for i, fw in enumerate(french_words):
    row = '  '.join(f'{attn_weights[i,j]:6.3f}' for j in range(len(english_words)))
    print(f'{fw:>8} {row}')

In [ ]:
# Visualize the attention weights as a heatmap

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(attn_weights, cmap='Blues', aspect='auto')

ax.set_xticks(range(len(english_words)))
ax.set_xticklabels(english_words, fontsize=13)
ax.set_yticks(range(len(french_words)))
ax.set_yticklabels(french_words, fontsize=13)
ax.set_xlabel('English (source)', fontsize=13)
ax.set_ylabel('French (target)', fontsize=13)
ax.set_title('Attention Weights: Which English word does each French word look at?', fontsize=13)

# Add text annotations
for i in range(len(french_words)):
    for j in range(len(english_words)):
        val = attn_weights[i, j]
        color = 'white' if val > 0.4 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print('Key insight: "noir" attends strongly to "black", "chat" attends to "cat", etc.')
print('The model learns to directly connect related words regardless of position.')

## 5. Attention Step by Step

Let's now build attention from scratch for a single sentence, to see exactly how it works.

The **scaled dot-product attention** formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- **Q** (Query): "What am I looking for?"
- **K** (Key): "What do I contain?"
- **V** (Value): "What information do I provide?"
- **d_k**: dimension of keys (used for scaling)

Think of it like a search engine:
- Each word creates a **query** ("I'm looking for an adjective that modifies me")
- Each word creates a **key** ("I'm an adjective")
- Each word creates a **value** ("here's my actual information")
- The query-key dot product determines relevance
- The output is a weighted sum of values

In [ ]:
# Self-attention on a single sentence

sentence = ["The", "cat", "sat", "on", "the", "mat"]
n = len(sentence)
d_model = 8  # embedding dimension
d_k = 4      # query/key dimension

# Step 1: Start with token embeddings
np.random.seed(42)
X = np.random.randn(n, d_model)  # (6, 8) — one row per word

# Step 2: Create Q, K, V projection matrices
W_Q = np.random.randn(d_model, d_k) * 0.3
W_K = np.random.randn(d_model, d_k) * 0.3
W_V = np.random.randn(d_model, d_k) * 0.3

# Step 3: Project
Q = X @ W_Q  # (6, 4)
K = X @ W_K  # (6, 4)
V = X @ W_V  # (6, 4)

print("Shapes:")
print(f"  X (input):   {X.shape}")
print(f"  Q (queries): {Q.shape}")
print(f"  K (keys):    {K.shape}")
print(f"  V (values):  {V.shape}")

# Step 4: Compute attention scores
scores = Q @ K.T / np.sqrt(d_k)  # (6, 6) — every word vs every word
print(f"\nRaw scores shape: {scores.shape}")

# Step 5: Softmax
weights = softmax(scores)

# Step 6: Weighted sum of values
output = weights @ V  # (6, 4)
print(f"Output shape: {output.shape}")
print("\nEach word now has a new representation that blends info from all other words!")

In [ ]:
# Visualize the self-attention weights

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap='Purples', aspect='auto')

ax.set_xticks(range(n))
ax.set_xticklabels(sentence, fontsize=13)
ax.set_yticks(range(n))
ax.set_yticklabels(sentence, fontsize=13)
ax.set_xlabel('Attending to (keys)', fontsize=12)
ax.set_ylabel('Query word', fontsize=12)
ax.set_title('Self-Attention Weights', fontsize=14)

for i in range(n):
    for j in range(n):
        val = weights[i, j]
        color = 'white' if val > 0.3 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print("Each row sums to 1.0 (it's a probability distribution).")
print("Row i shows: when processing word i, how much attention goes to each other word.")

### Why Scale by $\sqrt{d_k}$?

Without scaling, when `d_k` is large, dot products tend to be large in magnitude, pushing softmax into regions with very small gradients. Let's see this:

In [ ]:
# Effect of scaling on softmax

d_k_values = [4, 64, 512]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for ax, dk in zip(axes, d_k_values):
    # Random queries and keys
    q = np.random.randn(dk)
    K_demo = np.random.randn(10, dk)
    
    raw = q @ K_demo.T
    scaled = raw / np.sqrt(dk)
    
    ax.bar(range(10), softmax(raw), alpha=0.5, label='Unscaled', color='red')
    ax.bar(range(10), softmax(scaled), alpha=0.5, label='Scaled', color='blue')
    ax.set_title(f'd_k = {dk}')
    ax.set_ylabel('Attention weight')
    ax.legend(fontsize=9)

plt.suptitle('Scaling prevents softmax from becoming too peaky', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("Without scaling, large d_k makes softmax almost one-hot (all attention on one word).")
print("Scaling keeps the distribution smooth so gradients can flow.")

## Exercises

### Exercise 1: Attention with a Meaningful Sentence

Create embeddings for the sentence `["I", "love", "deep", "learning"]` where:
- "deep" and "learning" have similar embeddings (they're related)
- "I" and "love" have similar embeddings (subject-verb)

Compute self-attention and plot the heatmap. Do the attention weights reflect the relationships you encoded?

In [ ]:
# Exercise 1: Your code here
# Hints:
# 1. Create a (4, d) embedding matrix
# 2. Make rows 2,3 similar and rows 0,1 similar (e.g., add a shared vector)
# 3. Compute Q, K, V with random projection matrices
# 4. Compute scores, softmax, and plot



### Exercise 2: Masked (Causal) Attention

In language models like GPT, each word can only attend to **previous** words (you can't peek at the future when generating text). This is done by setting future positions to $-\infty$ before softmax.

Modify the self-attention code above to apply a **causal mask**:
```
mask = [[0,   -inf, -inf, -inf],
        [0,    0,   -inf, -inf],
        [0,    0,    0,   -inf],
        [0,    0,    0,    0  ]]
```
Add this mask to the scores before softmax and visualize the resulting attention weights.

In [ ]:
# Exercise 2: Your code here
# Hints:
# 1. Use np.triu(np.ones((n, n)), k=1) to create upper-triangle of ones
# 2. Multiply by -1e9 (a large negative number, acts like -inf)
# 3. Add to scores before softmax
# 4. Verify that attention weights in the upper triangle are ~0



## 6. What the Transformer Paper Proposes

In 2017, the paper *"Attention Is All You Need"* (Vaswani et al.) proposed a radical idea:

> **Throw away recurrence entirely. Use ONLY attention.**

The Transformer architecture has these key components:

```
  Input Embeddings
       |
       + Positional Encoding     ← "inject" position info (since no recurrence)
       |
  ┌────────────────┐
  │  Multi-Head     │
  │  Self-Attention  │           ← attend to all positions in parallel
  │                  │
  │  + Add & Norm    │           ← residual connection + layer norm
  │                  │
  │  Feed-Forward    │           ← process each position independently
  │  Network         │
  │                  │
  │  + Add & Norm    │
  └────────────────┘
       × N layers                ← stack this block N times
       |
     Output
```

Key innovations:

1. **Self-attention** replaces recurrence — process all positions in parallel
2. **Positional encoding** — since there's no recurrence, add position info explicitly
3. **Multi-head attention** — run attention multiple times in parallel (different "perspectives")
4. **Residual connections** — help gradients flow through deep networks
5. **Layer normalization** — stabilize training

### Why It Works Better

| Property | RNN | Transformer |
|----------|-----|-------------|
| Sequential ops | O(n) | O(1) |
| Max path length | O(n) | O(1) |
| Parallelizable | No | Yes |
| Long-range deps | Hard | Easy |

The "max path length" row is crucial: in an RNN, information from token 1 must pass through all intermediate states to reach token 100. In a Transformer, token 1 and token 100 are directly connected through attention — the path length is 1.

We will build every one of these components from scratch in the coming notebooks.

## Summary

In this notebook we learned:

- **Sequences** (text, time series) require models that respect order
- **RNNs** process tokens one at a time with a hidden state — clever, but slow and forgetful
- **Three RNN problems**: sequential bottleneck, vanishing gradients, fixed-size memory
- **Attention** lets the model look at all positions simultaneously and decide what's relevant
- **Scaled dot-product attention**: compute similarity scores, apply softmax, take weighted sum
- **Transformers** use only attention (no recurrence), enabling full parallelism and direct long-range connections

---

## What's Next

In **Notebook 04**, we'll start building the Transformer architecture piece by piece:
- Token and positional embeddings
- Single-head self-attention (implemented from scratch)
- Multi-head attention

We now have the *why*. Next, we build the *how*.